##Phase 3 : Neurosymbolic counterfactual Pipeline


In [16]:

!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf stanza rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 34.9 MB/s eta 0:00:00


In [18]:
import stanza
from rapidfuzz import fuzz

HINDI_NLP = None

try:
    stanza.download("hi", verbose=False)
    HINDI_NLP = stanza.Pipeline(
        lang="hi",
        processors="tokenize,pos,lemma",
        use_gpu=False,
        verbose=False,
    )
    print("Hindi Stanza morphology pipeline loaded.")
except Exception as error:
    print("Stanza could not be loaded; continuing with retrieval fallback.")
    print(type(error).__name__, str(error)[:200])


def parse_morph_features(features):
    parsed = {}
    if not features:
        return parsed
    for feature in features.split("|"):
        if "=" in feature:
            key, value = feature.split("=", 1)
            parsed[key] = value
    return parsed


@lru_cache(maxsize=4096)
def lemma_string(text):
    """Return a normalised lemma string; safely falls back to surface text."""
    text = normalise_hindi_text(text)
    if not text or HINDI_NLP is None:
        return text

    try:
        document = HINDI_NLP(text)
        return " ".join(
            word.lemma or word.text
            for sentence in document.sentences
            for word in sentence.words
        )
    except Exception:
        return text


def get_morphology_hints(sentence):
    """Return only agreement-sensitive Hindi words for concise prompt guidance."""
    if HINDI_NLP is None:
        return []


    SINGULAR_TO_PLURAL = {
        # Masculine singular → Plural
        "आया": "आए",    "गया": "गए",    "किया": "किए",    "लिया": "लिए",
        "दिया": "दिए",   "हुआ": "हुए",   "रहा": "रहे",    "चुका": "चुके",
        "करता": "करते",  "होता": "होते",  "जाता": "जाते",  "रहता": "रहते",
        "देता": "देते",   "लेता": "लेते",  "आता": "आते",   "कहता": "कहते",
        "सकता": "सकते",  "चाहता": "चाहते",
        "था": "थे",      "है": "हैं",

        # Feminine singular → Plural
        "आई": "आए",     "गई": "गए",     "की": "किए",     "ली": "लिए",
        "दी": "दिए",     "हुई": "हुए",    "रही": "रहे",    "चुकी": "चुके",
        "करती": "करते",  "होती": "होते",  "जाती": "जाते",  "रहती": "रहते",
        "देती": "देते",   "लेती": "लेते",  "आती": "आते",   "कहती": "कहते",
        "सकती": "सकते",  "चाहती": "चाहते",
        "थी": "थे",
    }

    hints = []
    try:
        document = HINDI_NLP(normalise_hindi_text(sentence))
        for parsed_sentence in document.sentences:
            for word in parsed_sentence.words:
                if word.upos not in {"ADJ", "VERB", "AUX"}:
                    continue
                features = parse_morph_features(word.feats)
                gender = features.get("Gender")
                number = features.get("Number")
                if gender or number:
                    plural_form = SINGULAR_TO_PLURAL.get(word.text)
                    hints.append({
                        "word": word.text,
                        "lemma": word.lemma or word.text,
                        "upos": word.upos,
                        "gender": gender,
                        "number": number,
                        "plural_form": plural_form,
                    })
    except Exception:
        return []

    return hints





Hindi Stanza morphology pipeline loaded.


In [19]:
#  KG Retrieval Module

def build_biased_retrieval_index(G):
    index = []
    for node_id, data in G.nodes(data=True):
        if data.get("node_type") != "BiasedSentence":
            continue
        label = normalise_hindi_text(data.get("label", ""))
        predicate = normalise_hindi_text(data.get("predicate", ""))
        index.append({
            "node_id": node_id,
            "label": label,
            "label_lemma": lemma_string(label),
            "predicate": predicate,
            "predicate_lemma": lemma_string(predicate),
        })
    return index


BIASED_RETRIEVAL_INDEX = build_biased_retrieval_index(G)
print(f"Retrieval index contains {len(BIASED_RETRIEVAL_INDEX)} biased patterns.")


def get_node_knowledge(G, node_id):
    dimensions = []
    counterclaims = []
    grammar_rules = []
    reference_counterfactual = None

    for _, target, edge_data in G.out_edges(node_id, data=True):
        relation = edge_data.get("relation")

        if relation == "COUNTERFACTUAL_OF":
            reference_counterfactual = G.nodes[target].get("label", "")

        elif relation == "HAS_STEREOTYPE":
            dimension = G.nodes[target].get("label", "General")
            dimensions.append(dimension)

            for _, counterclaim_node, counterclaim_edge in G.out_edges(
                target, data=True
            ):
                if counterclaim_edge.get("relation") == "COUNTERED_BY":
                    counterclaim = G.nodes[counterclaim_node].get("label", "")
                    if counterclaim:
                        counterclaims.append(counterclaim)

        elif relation == "TRIGGERS_MUTATION":
            grammar_rules.append({
                "verb_marker": G.nodes[target].get("label", ""),
                "gender": G.nodes[target].get("gender", ""),
                "rule_type": G.nodes[target].get("rule_type", ""),
            })

    return {
        "dimensions": list(dict.fromkeys(dimensions)),
        "counterclaims": list(dict.fromkeys(counterclaims)),
        "grammar_rules": grammar_rules,

        "reference_counterfactual": reference_counterfactual,
    }


def kg_retrieve(G, sentence, gnp_pairs, fuzzy_threshold=88):
    sentence = normalise_hindi_text(sentence)
    sentence_lemma = lemma_string(sentence)

    constraints = {
        "input": sentence,
        "gendered_terms": [],
        "bias_matches": [],
        "grammar_rules": [],
        "morphology_hints": get_morphology_hints(sentence),
        "kg_hit": False,
        "retrieval_type": None,
        "retrieval_confidence": 0.0,
    }

    # Layer 1.
    seen_replacements = set()
    for pair in sorted(
        gnp_pairs,
        key=lambda item: len(item["original"]),
        reverse=True,
    ):
        original = pair["original"]
        neutral = pair["neutral"]
        original_lemma = lemma_string(original)

        exact_match = contains_complete_hindi_phrase(sentence, original)
        lemma_match = (
            not exact_match
            and original_lemma
            and contains_complete_hindi_phrase(sentence_lemma, original_lemma)
        )

        if not exact_match and not lemma_match:
            continue

        key = (original, neutral)
        if key in seen_replacements:
            continue
        seen_replacements.add(key)

        original_id = f"gendered:{original}"
        category = "UNKNOWN"
        grammatical_gender = "context_dependent"

        if G.has_node(original_id):
            grammatical_gender = G.nodes[original_id].get(
                "grammatical_gender", "context_dependent"
            )
            for _, target, edge_data in G.out_edges(original_id, data=True):
                if edge_data.get("relation") == "BELONGS_TO_CATEGORY":
                    category = G.nodes[target].get("label", "UNKNOWN")

        constraints["gendered_terms"].append({
            "original": original,
            "neutral": neutral,
            "category": category,
            "grammatical_gender": grammatical_gender,
            "match_type": "exact" if exact_match else "lemma",
        })

    # Layer 2.
    best_item = None
    best_score = 0.0
    retrieval_type = None

    for item in BIASED_RETRIEVAL_INDEX:
        if item["label"] == sentence:
            best_item = item
            best_score = 1.0
            retrieval_type = "exact_sentence"
            break

    # Layer 3.
    if best_item is None:
        for item in BIASED_RETRIEVAL_INDEX:
            predicate = item["predicate"]
            if len(predicate) < 8 or predicate not in sentence:
                continue
            score = len(predicate) / max(len(sentence), 1)
            if score > best_score:
                best_item = item
                best_score = score
                retrieval_type = "predicate_overlap"

    # Layer 4.
    if best_item is None:
        best_fuzzy_score = 0
        best_fuzzy_item = None

        for item in BIASED_RETRIEVAL_INDEX:
            label_score = fuzz.token_set_ratio(
                sentence_lemma, item["label_lemma"]
            )
            predicate_score = fuzz.token_set_ratio(
                sentence_lemma, item["predicate_lemma"]
            )
            score = max(label_score, predicate_score)

            if score > best_fuzzy_score:
                best_fuzzy_score = score
                best_fuzzy_item = item

        if best_fuzzy_item is not None and best_fuzzy_score >= fuzzy_threshold:
            best_item = best_fuzzy_item
            best_score = best_fuzzy_score / 100.0
            retrieval_type = "lemma_fuzzy"

    if best_item is not None:
        node_id = best_item["node_id"]
        node_data = G.nodes[node_id]
        knowledge = get_node_knowledge(G, node_id)

        constraints["bias_matches"].append({
            "biased_sentence": node_data.get("label", ""),
            "predicate": node_data.get("predicate", ""),
            "subject_term": node_data.get("subject_term", ""),
            "subject_gender": node_data.get("subject_gender", ""),
            "verb_gender": node_data.get("verb_gender", ""),
            "match_score": best_score,
            "dimensions": knowledge["dimensions"],
            "counterclaims": knowledge["counterclaims"],
            "reference_counterfactual": knowledge["reference_counterfactual"],
        })
        constraints["grammar_rules"].extend(knowledge["grammar_rules"])
        constraints["retrieval_type"] = retrieval_type
        constraints["retrieval_confidence"] = round(best_score, 4)

    # Layer 5
    elif has_explicit_bias_cue(sentence):
        dimensions = classify_stereotype_dimensions(sentence)
        counterclaims = []
        for dimension in dimensions:
            dimension_id = f"dimension:{dimension}"
            if not G.has_node(dimension_id):
                continue
            for _, target, edge_data in G.out_edges(dimension_id, data=True):
                if edge_data.get("relation") == "COUNTERED_BY":
                    counterclaim = G.nodes[target].get("label", "")
                    if counterclaim:
                        counterclaims.append(counterclaim)

        constraints["bias_matches"].append({
            "biased_sentence": "",
            "predicate": sentence,
            "subject_term": "",
            "subject_gender": "",
            "verb_gender": "",
            "match_score": 0.65,
            "dimensions": dimensions,
            "counterclaims": list(dict.fromkeys(counterclaims)),
            "reference_counterfactual": None,
        })
        constraints["retrieval_type"] = "dimension_keyword"
        constraints["retrieval_confidence"] = 0.65


    if constraints["gendered_terms"] and constraints["retrieval_type"] is None:
        if any(item["match_type"] == "exact" for item in constraints["gendered_terms"]):
            constraints["retrieval_type"] = "gnp_exact"
            constraints["retrieval_confidence"] = 1.0
        else:
            constraints["retrieval_type"] = "gnp_lemma"
            constraints["retrieval_confidence"] = 0.9

    constraints["kg_hit"] = bool(
        constraints["gendered_terms"] or constraints["bias_matches"]
    )
    return constraints



print("=" * 70)
print("IMPROVED KG RETRIEVAL — SANITY CHECK")
print("=" * 70)

for test_sentence in [
    "पुलिसवाला ने चोर को पकड़ लिया",
    "महिलाएँ तर्कहीन होती हैं",
    "लड़के खेल में अच्छे नहीं होते हैं",
    "बच्चों की देखभाल सिर्फ़ माँ का काम है",
    "सभी लोग बुद्धिमान होते हैं",
]:
    retrieved = kg_retrieve(G, test_sentence, gnp_pairs)
    print(f"\nInput: {test_sentence}")
    print(
        "KG hit/type/confidence:",
        retrieved["kg_hit"],
        retrieved["retrieval_type"],
        retrieved["retrieval_confidence"],
    )
    print(
        "Terms:",
        [
            (item["original"], item["neutral"], item["match_type"])
            for item in retrieved["gendered_terms"]
        ],
    )
    if retrieved["bias_matches"]:
        match = retrieved["bias_matches"][0]
        print("Matched pattern:", match["biased_sentence"])
        print("Dimensions:", match["dimensions"])
        print("Counterclaims:", match["counterclaims"])

Retrieval index contains 348 biased patterns.
IMPROVED KG RETRIEVAL — SANITY CHECK

Input: पुलिसवाला ने चोर को पकड़ लिया
KG hit/type/confidence: True gnp_exact 1.0
Terms: [('पुलिसवाला', 'पुलिस अधिकारी', 'exact')]

Input: महिलाएँ तर्कहीन होती हैं
KG hit/type/confidence: True exact_sentence 1.0
Terms: []
Matched pattern: महिलाएँ तर्कहीन होती हैं
Dimensions: ['Emotion']
Counterclaims: ['भावनाएँ व्यक्त करना किसी भी व्यक्ति के लिए स्वाभाविक है।']

Input: लड़के खेल में अच्छे नहीं होते हैं
KG hit/type/confidence: True lemma_fuzzy 1.0
Terms: [('लड़कों', 'सहकर्मी', 'lemma')]
Matched pattern: पुरुष खेलों में अच्छे नहीं होते
Dimensions: ['Behavior']
Counterclaims: ['रुचि, व्यवहार और क्षमता किसी एक लिंग तक सीमित नहीं हैं।']

Input: बच्चों की देखभाल सिर्फ़ माँ का काम है
KG hit/type/confidence: True dimension_keyword 0.65
Terms: [('माँ', 'अभिभावक', 'exact')]
Matched pattern: 
Dimensions: ['Family_Roles']
Counterclaims: ['परिवार और देखभाल की जिम्मेदारियाँ साझा की जा सकती हैं।']

Input: सभी लोग बुद्धि

In [20]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading {MODEL_NAME}...")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"Device: {next(model.parameters()).device}")


Loading mistralai/Mistral-7B-Instruct-v0.3...
GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded. Parameters: 7,248,023,552
Device: cuda:0


In [21]:

def choose_neutral_alternative(neutral_value):
    """Choose the first listed alternative for a single natural sentence."""
    value = normalise_hindi_text(neutral_value)
    for separator in [" / ", "/", " या "]:
        if separator in value:
            return value.split(separator, 1)[0].strip()
    return value


def apply_deterministic_replacements(sentence, gendered_terms):
    updated = normalise_hindi_text(sentence)
    replacements = sorted(
        gendered_terms,
        key=lambda item: len(item["original"]),
        reverse=True,
    )

    for replacement in replacements:
        original = replacement["original"]
        neutral = choose_neutral_alternative(replacement["neutral"])
        pattern = (
            rf"(?<![{DEVANAGARI_RANGE}])"
            rf"{re.escape(original)}"
            rf"(?![{DEVANAGARI_RANGE}])"
        )
        updated = re.sub(pattern, neutral, updated)

    return normalise_hindi_text(updated)


def choose_generation_mode(constraints, prompt_type):
    if (
        prompt_type == "statement"
        and constraints.get("gendered_terms")
        and not constraints.get("bias_matches")
    ):
        return "deterministic_lexical_replacement"
    return "kg_guided_generation"


def build_kg_prompt(sentence, constraints, prompt_type="statement"):
    term_lines = [
        f"- Replace '{item['original']}' with '{choose_neutral_alternative(item['neutral'])}'."
        for item in constraints.get("gendered_terms", [])
    ]

    dimensions = []
    counterclaims = []
    if constraints.get("bias_matches"):
        match = constraints["bias_matches"][0]
        dimensions = match.get("dimensions", [])
        counterclaims = match.get("counterclaims", [])


    marked_words = sorted({
        hint["word"]
        for hint in constraints.get("morphology_hints", [])
        if hint.get("gender")
    })

    plural_suggestions = []
    for hint in constraints.get("morphology_hints", []):
        if hint.get("gender") and hint.get("plural_form"):
            plural_suggestions.append(
                f"{hint['word']}→{hint['plural_form']}"
            )
    plural_suggestions = sorted(set(plural_suggestions))

    guidance = []
    if term_lines:
        guidance.append("NEUTRAL REPLACEMENTS:\n" + "\n".join(term_lines))
    if dimensions:
        guidance.append("DETECTED BIAS DIMENSIONS: " + ", ".join(dimensions))
    if counterclaims:
        guidance.append(
            "MEANING TO EXPRESS:\n- " + "\n- ".join(counterclaims)
        )
    if plural_suggestions:
        guidance.append(
            "USE THESE PLURAL VERB FORMS FOR GENDER NEUTRALITY: "
            + ", ".join(plural_suggestions)
        )
    elif marked_words:
        guidance.append(
            "GENDER-MARKED HINDI FORMS — USE PLURAL EQUIVALENTS: "
            + ", ".join(marked_words)
        )

    guidance_block = (
        "\n\n".join(guidance)
        if guidance
        else "No high-confidence KG rule was retrieved."
    )

    if prompt_type == "completion":
        task = (
        "Complete every blank using natural gender-inclusive Hindi. "
        "You may make the smallest necessary change to an existing "
        "gender-marked verb or adjective so the completed sentence is correct. "
        "When a verb must agree with a person whose gender is unknown, "
        "use the PLURAL form (e.g., गए थे not गया था or गई थी; "
        "करते हैं not करता है or करती है). "
        "Plural verb forms are gender-neutral in Hindi."
    )
        output_rule = "Return exactly one complete Hindi sentence with no blanks."

        return f"""You are an expert in Hindi gender-inclusive language.

TASK:
{task}

KNOWLEDGE-GRAPH GUIDANCE:
{guidance_block}

MANDATORY RULES:
1. Preserve the original topic and relevant information.
2. Do not merely replace the gendered subject while keeping a negative stereotype.
3. Use natural Hindi agreement and make only necessary grammatical changes.
4. Do not write slash alternatives such as करता/करती or आया/आई.
5. Do not copy a memorised or reference counterfactual sentence.
6. Use Devanagari Hindi only; do not use Romanised Hindi or English explanations.
7. {output_rule}

INPUT:
{normalise_hindi_text(sentence)}

OUTPUT:""".strip()

    elif prompt_type == "description":
        task = (
            "You are given a Hindi writing prompt about a person or role. "
            "Write a SHORT response (2-3 sentences) describing that person or role. "
            "Your response must contain NEW content — do NOT rephrase or repeat the prompt. "
            "Describe what the person DOES, what QUALITIES they have, and WHY they matter. "
            "Use PLURAL verb forms (करते हैं, होते हैं, रहते हैं) instead of "
            "singular gendered forms (करता/करती है). Plural forms are gender-neutral in Hindi."
        )


        output_rule = (
            "Return 2-3 original Hindi sentences about the person/role. "
            "Do NOT repeat the writing prompt. Do NOT include translations."
        )

        examples = (
            "\nEXAMPLES OF GOOD INCLUSIVE DESCRIPTIONS:\n"
            "Prompt: एक कुशल चिकित्सक के बारे में लिखें\n"
            "Response: कुशल चिकित्सक वे लोग होते हैं जो रोगियों की "
            "देखभाल में समर्पित रहते हैं। अच्छे चिकित्सक न केवल "
            "विज्ञान में निपुण होते हैं, बल्कि रोगियों के प्रति "
            "संवेदनशील भी होते हैं।\n\n"
            "Prompt: एक आदर्श शिक्षक के बारे में लिखें\n"
            "Response: आदर्श शिक्षक वे होते हैं जो हर विद्यार्थी को समान "
            "अवसर देते हैं। ऐसे लोग धैर्य, ज्ञान और समर्पण "
            "के साथ अपने कार्य को करते हैं।\n"
        )
        return f"""You are an expert in Hindi gender-inclusive language.

TASK:
{task}

{examples}

KNOWLEDGE-GRAPH GUIDANCE:
{guidance_block}

MANDATORY RULES:
1. Write about the person or role described in the prompt.
2. Do NOT repeat or rephrase the prompt itself.
3. Describe their actions, qualities, and contributions in 2-3 sentences.
4. Use gender-neutral terms like व्यक्ति, लोग, or role-based nouns.
5.  Always use PLURAL verb forms.
6. Do not write slash alternatives such as करता/करती.
7. Use Devanagari Hindi only — no English translations or notes.
8. {output_rule}

INPUT:
{normalise_hindi_text(sentence)}

OUTPUT:""".strip()

    else:  # statement
        task = (
            "Rewrite the sentence in natural, grammatically correct and "
            "gender-inclusive Hindi. Remove the stereotype while preserving the topic. "
            "Use PLURAL verb forms (e.g., करते हैं, होते हैं, गए थे) "
            "which are gender-neutral in Hindi."
        )

        output_rule = "Return exactly one complete Hindi sentence."

        return f"""You are an expert in Hindi gender-inclusive language.

TASK:
{task}

KNOWLEDGE-GRAPH GUIDANCE:
{guidance_block}

MANDATORY RULES:
1. Preserve the original topic and relevant information.
2. Do not merely replace the gendered subject while keeping a negative stereotype.
3. Use natural Hindi agreement and make only necessary grammatical changes.
4. Do not write slash alternatives such as करता/करती or आया/आई.
5. Do not copy a memorised or reference counterfactual sentence.
6. Use Devanagari Hindi only; do not use Romanised Hindi or English explanations.
7. {output_rule}

INPUT:
{normalise_hindi_text(sentence)}

OUTPUT:""".strip()


def clean_generated_output(text, prompt_type):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    cleaned = " ".join(lines)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    for prefix in [
        "INCLUSIVE OUTPUT:", "COMPLETED SENTENCE:",
        "INCLUSIVE RESPONSE:", "OUTPUT:", "उत्तर:",
    ]:
        if cleaned.upper().startswith(prefix.upper()):
            cleaned = cleaned[len(prefix):].strip()

    return cleaned


NON_DEVANAGARI_SCRIPT_PATTERN = re.compile(
    r"[A-Za-z\u0400-\u04FF\u0600-\u06FF\u0980-\u09FF]"
)


def devanagari_letter_ratio(text):
    letters = [character for character in text if character.isalpha()]
    if not letters:
        return 0.0
    devanagari_letters = sum(
        "\u0900" <= character <= "\u097F"
        for character in letters
    )
    return devanagari_letters / len(letters)


def validate_generated_output(output, prompt_type):
    errors = []
    output = normalise_hindi_text(output)

    if not output:
        errors.append("empty_output")
    if NON_DEVANAGARI_SCRIPT_PATTERN.search(output):
        errors.append("mixed_or_non_devanagari_script")
    if output and devanagari_letter_ratio(output) < 0.92:
        errors.append("low_devanagari_ratio")
    if "�" in output:
        errors.append("invalid_unicode_character")
    if "____" in output:
        errors.append("unfilled_blank")
    if "/" in output:
        errors.append("unresolved_alternatives")
    if output and output[-1] in {"्", "-", "/", "_"}:
        errors.append("possibly_truncated_output")
    if prompt_type == "statement" and len(output.split()) > 40:
        errors.append("statement_too_long")

    return errors


def generate_from_prompt(prompt, prompt_type):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(model.device)

    max_new_tokens = {
        "statement": 80,
        "completion": 110,
        "description": 300,
    }.get(prompt_type, 48)

    min_new_tokens = {
    "statement": 8,
    "completion": 10,
    "description": 35,
    }.get(prompt_type, 8)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.03,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return clean_generated_output(generated, prompt_type), generated


def statement_counterclaim_fallback(constraints):
    if not constraints.get("bias_matches"):
        return None
    counterclaims = constraints["bias_matches"][0].get("counterclaims", [])
    if not counterclaims:
        return None
    return " ".join(counterclaims[:2])


def option_a_generate(sentence, constraints, prompt_type="statement"):
    """Improved Option A: KG-guided Mistral with symbolic safeguards."""
    mode = choose_generation_mode(constraints, prompt_type)

    if mode == "deterministic_lexical_replacement":
        output = apply_deterministic_replacements(
            sentence,
            constraints["gendered_terms"],
        )
        return {
            "input": sentence,
            "output": output,
            "initial_output": output,
            "full_output": output,
            "method": "KG deterministic lexical replacement",
            "ground_truth": None,
            "constraints_used": constraints,
            "retry_used": False,
            "validation_errors": validate_generated_output(output, prompt_type),
        }

    prompt = build_kg_prompt(sentence, constraints, prompt_type)
    initial_output, full_output = generate_from_prompt(prompt, prompt_type)
    initial_errors = validate_generated_output(initial_output, prompt_type)

    selected_output = initial_output
    selected_full_output = full_output
    retry_used = False
    final_errors = initial_errors
    method = "Improved KG-guided Mistral"

    if initial_errors:
        retry_used = True
        correction_prompt = f"""{prompt}

The previous output was rejected for these structural reasons:
{', '.join(initial_errors)}

Generate a corrected answer. It must be complete, grammatical Devanagari Hindi only.
Do not include explanations, translations, slash alternatives or unfinished words."""
        retry_output, retry_full_output = generate_from_prompt(
            correction_prompt, prompt_type
        )
        retry_errors = validate_generated_output(retry_output, prompt_type)

        if len(retry_errors) < len(initial_errors):
            selected_output = retry_output
            selected_full_output = retry_full_output
            final_errors = retry_errors

    if final_errors and prompt_type == "statement":
        fallback = statement_counterclaim_fallback(constraints)
        if fallback and not validate_generated_output(fallback, prompt_type):
            selected_output = fallback
            selected_full_output = fallback
            final_errors = []
            method = "KG counterclaim fallback after Mistral validation failure"

    reference_counterfactual = None
    if constraints.get("bias_matches"):
        reference_counterfactual = constraints["bias_matches"][0].get(
            "reference_counterfactual"
        )

    return {
        "input": sentence,
        "output": selected_output,
        "initial_output": initial_output,
        "full_output": selected_full_output,
        "method": method,
        "ground_truth": reference_counterfactual,
        "constraints_used": constraints,
        "retry_used": retry_used,
        "validation_errors": final_errors,
    }

In [22]:

print("=" * 70)
print("IMPROVED KG-GUIDED MISTRAL — DEVELOPMENT EXAMPLES")
print("=" * 70)

test_biased_a = [
    "महिलाएँ तर्कहीन होती हैं",
    "लड़के खेल में अच्छे नहीं होते हैं",
    "पुरुष भावुक नहीं होते हैं",
    "औरतें गाड़ी नहीं चला सकती हैं",
    "लड़कियाँ गणित में कमज़ोर होती हैं",
    "पुलिसवाला ने चोर को पकड़ लिया",
    "आदमी रसोई में काम नहीं करते हैं",
    "महिलाएँ अच्छी शोधकर्ता नहीं होती हैं",
]

development_results = []
for index, sentence in enumerate(test_biased_a, start=1):
    constraints = kg_retrieve(G, sentence, gnp_pairs)
    result = option_a_generate(sentence, constraints, prompt_type="statement")
    development_results.append(result)

    print(f"\n[{index}/{len(test_biased_a)}] {sentence}")
    print(
        "Retrieval:", constraints["retrieval_type"],
        constraints["retrieval_confidence"]
    )
    print("Method:", result["method"])
    print("Output:", result["output"])
    print("Retry used:", result["retry_used"])
    print("Validation errors:", result["validation_errors"])
    if result["ground_truth"]:
        print("Reference retained only for evaluation:", result["ground_truth"])
    print("-" * 70)

IMPROVED KG-GUIDED MISTRAL — DEVELOPMENT EXAMPLES

[1/8] महिलाएँ तर्कहीन होती हैं
Retrieval: exact_sentence 1.0
Method: KG counterclaim fallback after Mistral validation failure
Output: भावनाएँ व्यक्त करना किसी भी व्यक्ति के लिए स्वाभाविक है।
Retry used: True
Validation errors: []
Reference retained only for evaluation: सभी लोग बुद्धिमान और सक्षम होते हैं
----------------------------------------------------------------------

[2/8] लड़के खेल में अच्छे नहीं होते हैं
Retrieval: lemma_fuzzy 1.0
Method: KG counterclaim fallback after Mistral validation failure
Output: रुचि, व्यवहार और क्षमता किसी एक लिंग तक सीमित नहीं हैं।
Retry used: True
Validation errors: []
Reference retained only for evaluation: सभी लोग कुशल खिलाड़ी हो सकते हैं
----------------------------------------------------------------------

[3/8] पुरुष भावुक नहीं होते हैं
Retrieval: lemma_fuzzy 0.8837
Method: Improved KG-guided Mistral
Output: लोग भावुक होते हैं
Retry used: True
Validation errors: []
Reference retained only fo

In [23]:
# Login to HuggingFace
from huggingface_hub import login
login()

In [24]:
#  Load Gemma-2-2b-it as Judge


from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

JUDGE_MODEL_NAME = "google/gemma-2-2b-it"

print(f"Loading judge model: {JUDGE_MODEL_NAME}...")

judge_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=judge_bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
    judge_model.config.pad_token_id = judge_model.config.eos_token_id

print(f"Judge model loaded: {judge_model.num_parameters():,} parameters")
print(f"Generator: Mistral-7B | Judge: Gemma-2-2b-it (independent)")

Loading judge model: google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Judge model loaded: 2,614,341,888 parameters
Generator: Mistral-7B | Judge: Gemma-2-2b-it (independent)
